In [1]:
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions

#### Antioxidant protein dataset preprocessing and metadata generation

This script processes antioxidant protein data from multiple files within a single source, parses custom-formatted sequence entries, handles duplicated sequences, and generates a clean dataset along with metadata for downstream classification tasks.

- Overview
    - Task: Antioxidant protein classification dataset preparation
    - Source: AOD database
    - Input: Multiple raw text files containing protein sequences and metadata Excel file
    - Output: Processed dataset (CSV) and metadata file (JSON)
- Process:
    - Read and parse protein sequences from multiple files in a directory
    - Extract sequence identifiers and corresponding sequences from custom-formatted entries
    - Remove identifier column and assign positive labels to all sequences
    - Check for duplicated sequences and label consistency
    - Concatenate consistent duplicates with unique sequences
    - Load and organize source metadata
    - Export processed data and metadata

- Auxiliary variables

In [2]:
path_export = "../../processed_dataset"
path_input = "../../raw_dataset"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "AOD"

- Read docs

In [3]:
records = []
folder_path = f"{path_input}/{name_source}"

for filename in os.listdir(folder_path):
    filepath = os.path.join(folder_path, filename)
    with open(filepath, 'r') as f:
        content = f.read()

    entries = content.strip().split('"\n">')
    for entry in entries:
        entry = entry.strip('"').strip('>')
        lines = entry.split('\n')
        protein_id = lines[0].strip()
        sequence = ''.join(lines[1:]).strip().strip('"')
        records.append({'id': protein_id, 'sequence': sequence})

df_data = pd.DataFrame(records)
print(df_data.shape)

(710, 2)


In [4]:
df_data = df_data.drop(columns=["id"])
df_data['label'] = 1
df_data

,sequence,label
0,MASELAMSNSDLPTSPLAMEYVNDFDLMKFEVKKEPVETDRIISQC...,1
1,MPGGLLLGDEAPNFEANTTIGRIRFHDYLGDSWGILFSHPRDFTPV...,1
2,MAATNTILAFSSPSRLLIPPSSNPSTLRSSFRGVSLNNNNLHRLQS...,1
3,MAQTVVLKVGMSCQGCVGAVNRVLGKMEGVESFDIDIKEQKVTVKG...,1
4,MVKAVCVVRGDSKVTGSIVFEQESESAPTKITWDISGNDANAKRGM...,1
...,...,...
705,MSLIGKEVLPFEAKAFKNGEFIDVTNEDLKGQWSVFCFYPADFSFV...,1
706,MATLKAVCVMKGDAPVEGVIHFQQQGSGPVKVTGKITGLSDGDHGF...,1
707,DARARSFVARAAAEYDLPLVGNKAPDFAAEAVFDQEFINVKLSDYI...,1
708,HSDLPSGVYDPAQA,1


- Checking duplicates

In [5]:
df_consistent_duplicates, df_errors, df_unique = ParsersCommons.processing_duplicated(
    df_data, group_seq= "sequence",
    label_col= "label")
df_consistent_duplicates.shape, df_errors.shape, df_unique.shape

((34, 3), (0, 0), (617, 2))

In [6]:
data_correct = pd.concat([df_consistent_duplicates, df_unique], axis=0, ignore_index=True)
data_correct = data_correct.drop(columns=["n_duplicates"])
data_correct.head()

,sequence,label
0,INGDAKGTVFFEQETSEAPVKVTGEGLGLAKGLHGFHVHEFGDNTN...,1
1,MAFAVSTACRPSLLLPPRQRSSPPRPRPLLCTPSTAAFRRGALSAT...,1
2,MALAVRVVYCGAUGYKPKYLQLKEKLEHEFPGCLDICGEGTPQVTG...,1
3,MALAVRVVYCGAUGYKSKYLQLKKKLEDEFPGRLDICGEGTPQATG...,1
4,MAMKAVCVLKGDSPVQGTINFEQKESNGPVKVWGSITGLTEGLHGF...,1


- Reading metadata

In [7]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
4,20260408032404.xls,AOD,Database,Static,No information,No,2017,2017-08-07,2026-04-07,fasta,Sequence,Enzyme/protein classification,Antioxidant,Sampling from Swiss-Prot,No information,http://lin-group.cn/AODdatabase/index.aspx,https://www.nature.com/articles/s41598-017-081...,No information
5,20260408032416.xls,AOD,Database,Static,No information,No,2017,2017-08-07,2026-04-07,fasta,Sequence,Enzyme/protein classification,Antioxidant,Sampling from Swiss-Prot,No information,http://lin-group.cn/AODdatabase/index.aspx,https://www.nature.com/articles/s41598-017-081...,No information
6,20260408032424.xls,AOD,Database,Static,No information,No,2017,2017-08-07,2026-04-07,fasta,Sequence,Enzyme/protein classification,Antioxidant,Sampling from Swiss-Prot,No information,http://lin-group.cn/AODdatabase/index.aspx,https://www.nature.com/articles/s41598-017-081...,No information
7,20260408032435.xls,AOD,Database,Static,No information,No,2017,2017-08-07,2026-04-07,fasta,Sequence,Enzyme/protein classification,Antioxidant,Sampling from Swiss-Prot,No information,http://lin-group.cn/AODdatabase/index.aspx,https://www.nature.com/articles/s41598-017-081...,No information
8,20260408032443.xls,AOD,Database,Static,No information,No,2017,2017-08-07,2026-04-07,fasta,Sequence,Enzyme/protein classification,Antioxidant,Sampling from Swiss-Prot,No information,http://lin-group.cn/AODdatabase/index.aspx,https://www.nature.com/articles/s41598-017-081...,No information


In [8]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': '20260408032404.xls;20260408032416.xls;20260408032424.xls;20260408032435.xls;20260408032443.xls;20260408032453.xls;20260408032500.xls;20260408032510.xls;20260408032515.xls;20260408032521.xls;20260408032542.xls;20260408032548.xls;20260408032553.xls;20260408032557.xls;20260408032603.xls;20260408032627.xls;20260408032635.xls;20260408032641.xls;20260408032646.xls;20260408032656.xls;20260408032716.xls;20260408032721.xls;20260408032725.xls;20260408032732.xls;20260408032737.xls;20260408032759.xls;20260408032803.xls;20260408032808.xls;20260408032812.xls;20260408032816.xls;20260408032830.xls;20260408032834.xls;20260408032839.xls;20260408032844.xls;20260408032849.xls;20260408032854.xls',
 'name source': 'AOD',
 'type source': 'Database',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'No',
 'year of publication': 2017,
 'last update date': Timestamp('2017-08-07 00:00:00'),
 'download date': Timestamp('2026-04-07 00:00:00'),
 'file format'

In [9]:
dict_metadata['number_of_records'] = df_data.shape[0]
dict_metadata['number_of_collected_sequences'] = df_data.shape[0]
dict_metadata['number_of_unique_sequences'] = data_correct.shape[0]
dict_metadata['positive_examples'] = data_correct[data_correct["label"] == 1].shape[0]
dict_metadata['negative_examples'] = data_correct[data_correct["label"] == 0].shape[0]
dict_metadata['number_of_sequences_with_errors'] = df_errors.shape[0]
dict_metadata

{'name dataset': '20260408032404.xls;20260408032416.xls;20260408032424.xls;20260408032435.xls;20260408032443.xls;20260408032453.xls;20260408032500.xls;20260408032510.xls;20260408032515.xls;20260408032521.xls;20260408032542.xls;20260408032548.xls;20260408032553.xls;20260408032557.xls;20260408032603.xls;20260408032627.xls;20260408032635.xls;20260408032641.xls;20260408032646.xls;20260408032656.xls;20260408032716.xls;20260408032721.xls;20260408032725.xls;20260408032732.xls;20260408032737.xls;20260408032759.xls;20260408032803.xls;20260408032808.xls;20260408032812.xls;20260408032816.xls;20260408032830.xls;20260408032834.xls;20260408032839.xls;20260408032844.xls;20260408032849.xls;20260408032854.xls',
 'name source': 'AOD',
 'type source': 'Database',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'No',
 'year of publication': 2017,
 'last update date': Timestamp('2017-08-07 00:00:00'),
 'download date': Timestamp('2026-04-07 00:00:00'),
 'file format'

- Export data

In [10]:
UtilsFunctions.make_directory(f"{path_export}/{name_task}/{name_source}")
UtilsFunctions.export_json(f"{path_export}/{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)
data_correct.to_csv(f"{path_export}/{name_task}/{name_source}/processed_data.csv", index=False)